# 多智能体系统
代理是一种使用 LLM 来决定应用程序控制流的系统。随着这些系统的开发，它们可能会随着时间的推移变得更加复杂，从而更难以管理和扩展。例如，您可能会遇到以下问题：

- 代理可以使用的工具太多，并且对于下一步调用哪个工具做出了错误的决定
- 环境变得过于复杂，单个代理无法跟踪
- 系统中需要多个专业领域（例如规划师、研究员、数学专家等）
- 为了解决这些问题，您可以考虑将应用程序拆分成多个较小的独立代理，并将它们组合成一个多代理系统。这些独立代理可以像提示符和 LLM 调用一样简单，也可以像ReAct代理一样复杂（甚至更多！）。

使用多代理系统的主要好处是：

- 模块化：独立的代理使得代理系统的开发、测试和维护变得更加容易。
- 专业化：您可以创建专注于特定领域的专家代理，这有助于提高整体系统性能。
- 控制：您可以明确控制代理如何通信（而不是依赖于函数调用）。

## 多代理架构

<img src="https://langchain-ai.github.io/langgraph/concepts/img/multi_agent/architectures.png">

在多代理系统中，有几种连接代理的方法：

- 网络：每个代理都可以与其他代理通信。任何代理都可以决定接下来要呼叫哪个代理。
- 主管代理：每个代理只与一个主管代理进行通信。主管代理负责决定接下来应该调用哪个代理。
- 主管（工具调用）：这是主管架构的一个特例。单个代理可以表示为工具。在这种情况下，主管代理使用工具调用 LLM 来决定调用哪些代理工具，以及传递给这些代理的参数。
- 分层结构：你可以定义一个多智能体系统，其中包含多个主管的主管。这是主管架构的泛化，允许更复杂的控制流。
- 自定义多代理工作流：每个代理仅与一部分代理进行通信。流程的某些部分是确定性的，只有部分代理可以决定接下来要调用哪些其他代理。

## 交接¶
在多智能体架构中，智能体可以表示为图节点。每个智能体节点执行其步骤，并决定是完成执行还是路由至其他智能体，包括可能路由至自身（例如，循环运行）。多智能体交互中一种常见的模式是切换，即一个智能体将控制权移交给另一个智能体。切换允许您指定：

- 目的地：要导航到的目标代理（例如，要前往的节点的名称）
- 有效载荷：传递给该代理的信息（例如，状态更新）

为了在 LangGraph 中实现切换，代理节点可以返回 Command 对象，该对象允许您结合控制流和状态更新：

In [ ]:
def agent(state) -> Command[Literal["agent", "another_agent"]]:
    # the condition for routing/halting can be anything, e.g. LLM tool call / structured output, etc.
    goto = get_next_agent(...)  # 'agent' / 'another_agent'
    return Command(
        # Specify which agent to call next
        goto=goto,
        # Update the graph state
        update={"my_state_key": "my_state_value"}
    )

在更复杂的场景中，每个代理节点本身就是一个图（即子图），其中一个代理子图中的节点可能需要导航到不同的代理。例如，如果您有两个代理，alice和bob（父图中的子图节点），并且alice需要导航到bob，则可以graph=Command.PARENT在Command对象中设置：

In [ ]:
def some_node_inside_alice(state):
    return Command(
        goto="bob",
        update={"my_state_key": "my_state_value"},
        # specify which graph to navigate to (defaults to the current graph)
        graph=Command.PARENT,
    )

如果您需要支持使用以下方式进行通信的子图可视化，Command(graph=Command.PARENT)则需要将它们包装在带有Command注释的节点函数中：而不是这样：

`builder.add_node(alice)`



In [ ]:
def call_alice(state) -> Command[Literal["bob"]]:
    return alice.invoke(state)

builder.add_node("alice", call_alice)

## 交接工具¶
最常见的代理类型之一是工具调用代理。对于这类代理，一种常见的模式是将切换包装在工具调用中：

API 参考：工具

In [ ]:
from langchain_core.tools import tool

@tool
def transfer_to_bob():
    """Transfer to bob."""
    return Command(
        # name of the agent (node) to go to
        goto="bob",
        # data to send to the agent
        update={"my_state_key": "my_state_value"},
        # indicate to LangGraph that we need to navigate to
        # agent node in a parent graph
        graph=Command.PARENT,
    )

> 如果要使用返回 Command 的工具，可以使用预构建的 create_react_agent / ToolNode 组件，或者实现自己的逻辑：

```python
def call_tools(state):
    ...
    commands = [tools_by_name[tool_call["name"]].invoke(tool_call) for tool_call in tool_calls]
    return commands
```

## 网络¶
在此架构中，代理被定义为图节点。每个代理可以与其他代理通信（多对多连接），并可以决定接下来调用哪个代理。此架构非常适合那些代理层级不明确或代理调用顺序不明确的问题。

In [ ]:
from typing import Literal
from langchain_openai import ChatOpenAI
from langgraph.types import Command
from langgraph.graph import StateGraph, MessagesState, START, END

model = ChatOpenAI()

def agent_1(state: MessagesState) -> Command[Literal["agent_2", "agent_3", END]]:
    # you can pass relevant parts of the state to the LLM (e.g., state["messages"])
    # to determine which agent to call next. a common pattern is to call the model
    # with a structured output (e.g. force it to return an output with a "next_agent" field)
    response = model.invoke(...)
    # route to one of the agents or exit based on the LLM's decision
    # if the LLM returns "__end__", the graph will finish execution
    return Command(
        goto=response["next_agent"],
        update={"messages": [response["content"]]},
    )

def agent_2(state: MessagesState) -> Command[Literal["agent_1", "agent_3", END]]:
    response = model.invoke(...)
    return Command(
        goto=response["next_agent"],
        update={"messages": [response["content"]]},
    )

def agent_3(state: MessagesState) -> Command[Literal["agent_1", "agent_2", END]]:
    ...
    return Command(
        goto=response["next_agent"],
        update={"messages": [response["content"]]},
    )

builder = StateGraph(MessagesState)
builder.add_node(agent_1)
builder.add_node(agent_2)
builder.add_node(agent_3)

builder.add_edge(START, "agent_1")
network = builder.compile()

## 导师¶
在此架构中，我们将代理定义为节点，并添加一个主管节点 (LLM)，用于决定下一步应调用哪些代理节点。我们Command根据主管节点的决定将执行路由到适当的代理节点。此架构也非常适合并行运行多个代理或使用Map-Reduce模式。

API 参考：ChatOpenAI |命令| StateGraph |开始|结束

In [ ]:
from typing import Literal
from langchain_openai import ChatOpenAI
from langgraph.types import Command
from langgraph.graph import StateGraph, MessagesState, START, END

model = ChatOpenAI()

def supervisor(state: MessagesState) -> Command[Literal["agent_1", "agent_2", END]]:
    # you can pass relevant parts of the state to the LLM (e.g., state["messages"])
    # to determine which agent to call next. a common pattern is to call the model
    # with a structured output (e.g. force it to return an output with a "next_agent" field)
    response = model.invoke(...)
    # route to one of the agents or exit based on the supervisor's decision
    # if the supervisor returns "__end__", the graph will finish execution
    return Command(goto=response["next_agent"])

def agent_1(state: MessagesState) -> Command[Literal["supervisor"]]:
    # you can pass relevant parts of the state to the LLM (e.g., state["messages"])
    # and add any additional logic (different models, custom prompts, structured output, etc.)
    response = model.invoke(...)
    return Command(
        goto="supervisor",
        update={"messages": [response]},
    )

def agent_2(state: MessagesState) -> Command[Literal["supervisor"]]:
    response = model.invoke(...)
    return Command(
        goto="supervisor",
        update={"messages": [response]},
    )

builder = StateGraph(MessagesState)
builder.add_node(supervisor)
builder.add_node(agent_1)
builder.add_node(agent_2)

builder.add_edge(START, "supervisor")

supervisor = builder.compile()

### 主管（工具调用）¶
在这个监督器架构的变体中，我们定义了一个监督器代理，负责调用子代理。子代理作为工具暴露给监督器，监督器代理决定下一步调用哪个工具。监督器代理遵循一个标准实现，即一个LLM，它在while循环中运行，调用各种工具，直到它决定停止。

API 参考：ChatOpenAI | InjectedState | create_react_agent


In [ ]:
from typing import Annotated
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import InjectedState, create_react_agent

model = ChatOpenAI()

# this is the agent function that will be called as tool
# notice that you can pass the state to the tool via InjectedState annotation
def agent_1(state: Annotated[dict, InjectedState]):
    # you can pass relevant parts of the state to the LLM (e.g., state["messages"])
    # and add any additional logic (different models, custom prompts, structured output, etc.)
    response = model.invoke(...)
    # return the LLM response as a string (expected tool response format)
    # this will be automatically turned to ToolMessage
    # by the prebuilt create_react_agent (supervisor)
    return response.content

def agent_2(state: Annotated[dict, InjectedState]):
    response = model.invoke(...)
    return response.content

tools = [agent_1, agent_2]
# the simplest way to build a supervisor w/ tool-calling is to use prebuilt ReAct agent graph
# that consists of a tool-calling LLM node (i.e. supervisor) and a tool-executing node
supervisor = create_react_agent(model, tools)

### 分层¶
随着系统中添加更多代理，主管可能会难以管理所有代理。主管可能会开始做出错误的决定，不知道接下来该呼叫哪个代理，或者情况可能变得过于复杂，单个主管无法跟踪。换句话说，你最终会遇到最初设计多代理架构时遇到的那些问题。

为了解决这个问题，您可以采用分层设计的方式设计系统。例如，您可以创建独立的、专门的代理团队，每个团队由不同的主管管理，并指定一个顶级主管来管理这些团队。

In [ ]:
from typing import Literal
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import Command
model = ChatOpenAI()

# define team 1 (same as the single supervisor example above)

def team_1_supervisor(state: MessagesState) -> Command[Literal["team_1_agent_1", "team_1_agent_2", END]]:
    response = model.invoke(...)
    return Command(goto=response["next_agent"])

def team_1_agent_1(state: MessagesState) -> Command[Literal["team_1_supervisor"]]:
    response = model.invoke(...)
    return Command(goto="team_1_supervisor", update={"messages": [response]})

def team_1_agent_2(state: MessagesState) -> Command[Literal["team_1_supervisor"]]:
    response = model.invoke(...)
    return Command(goto="team_1_supervisor", update={"messages": [response]})

team_1_builder = StateGraph(Team1State)
team_1_builder.add_node(team_1_supervisor)
team_1_builder.add_node(team_1_agent_1)
team_1_builder.add_node(team_1_agent_2)
team_1_builder.add_edge(START, "team_1_supervisor")
team_1_graph = team_1_builder.compile()

# define team 2 (same as the single supervisor example above)
class Team2State(MessagesState):
    next: Literal["team_2_agent_1", "team_2_agent_2", "__end__"]

def team_2_supervisor(state: Team2State):
    ...

def team_2_agent_1(state: Team2State):
    ...

def team_2_agent_2(state: Team2State):
    ...

team_2_builder = StateGraph(Team2State)
...
team_2_graph = team_2_builder.compile()


# define top-level supervisor

builder = StateGraph(MessagesState)
def top_level_supervisor(state: MessagesState) -> Command[Literal["team_1_graph", "team_2_graph", END]]:
    # you can pass relevant parts of the state to the LLM (e.g., state["messages"])
    # to determine which team to call next. a common pattern is to call the model
    # with a structured output (e.g. force it to return an output with a "next_team" field)
    response = model.invoke(...)
    # route to one of the teams or exit based on the supervisor's decision
    # if the supervisor returns "__end__", the graph will finish execution
    return Command(goto=response["next_team"])

builder = StateGraph(MessagesState)
builder.add_node(top_level_supervisor)
builder.add_node("team_1_graph", team_1_graph)
builder.add_node("team_2_graph", team_2_graph)
builder.add_edge(START, "top_level_supervisor")
builder.add_edge("team_1_graph", "top_level_supervisor")
builder.add_edge("team_2_graph", "top_level_supervisor")
graph = builder.compile()

## 自定义多代理工作流程¶
在这个架构中，我们将各个代理添加为图节点，并在自定义工作流中提前定义代理的调用顺序。在 LangGraph 中，工作流可以通过两种方式定义：

显式控制流（普通边） ：LangGraph 允许您通过普通图边显式定义应用程序的控制流（即代理通信的顺序）。这是上述架构中最具确定性的变体——我们始终能够提前知道下一个将被调用的代理。

动态控制流（命令）：在 LangGraph 中，您可以允许 LLM 决定应用程序控制流的某些部分。这可以通过使用 来实现Command。一个特殊的例子是主管工具调用架构。在这种情况下，驱动主管代理的工具调用 LLM 将决定工具（代理）的调用顺序。

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START

model = ChatOpenAI()

def agent_1(state: MessagesState):
    response = model.invoke(...)
    return {"messages": [response]}

def agent_2(state: MessagesState):
    response = model.invoke(...)
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node(agent_1)
builder.add_node(agent_2)
# define the flow explicitly
builder.add_edge(START, "agent_1")
builder.add_edge("agent_1", "agent_2")

## 通信和状态管理¶
构建多代理系统时最重要的是弄清楚代理如何通信。

代理之间沟通的一种常见方式是通过消息列表。这引出了以下问题：

- 代理是通过切换还是通过工具调用进行通信？
- 什么消息从一个代理传递到下一个代理？
- 交接在消息列表中如何表示？
- 您如何管理子代理的状态？

此外，如果您正在处理更复杂的代理或希望将单个代理状态与多代理系统状态分开，则可能需要使用不同的状态模式。

### 交接与工具调用¶
在代理之间传递的“有效载荷”是什么？在上面讨论的大多数架构中，代理通过切换进行通信，并将图状态作为切换有效载荷的一部分传递。具体来说，代理将消息列表作为图状态的一部分传递。对于具有工具调用功能的监督器，有效载荷是工具调用参数。

<img src="https://langchain-ai.github.io/langgraph/concepts/img/multi_agent/request.png">

### 代理之间的消息传递¶
代理之间最常见的通信方式是通过共享状态通道，通常是一串消息列表。这假设代理之间始终至少有一个状态通道（键）是共享的（例如messages）。通过共享消息列表进行通信时，还有一个额外的考虑：代理之间应该共享其思维过程的完整历史记录，还是仅共享最终结果？

<img src="https://langchain-ai.github.io/langgraph/concepts/img/multi_agent/response.png">

#### 分享完整的思考过程¶
代理可以与所有其他代理共享其思维过程的完整历史记录（即“暂存器”）。这个“暂存器”通常看起来像一个消息列表。共享完整思维过程的好处在于，它可以帮助其他代理做出更好的决策，并提高整个系统的推理能力。缺点是，随着代理数量及其复杂性的增长，“暂存器”也会快速增长，可能需要额外的内存管理策略。

#### 仅分享最终结果¶
代理可以拥有自己的私有“暂存器”，并且只与其他代理共享最终结果。这种方法可能更适合包含多个代理或代理较为复杂的系统。在这种情况下，您需要定义具有不同状态模式的代理。

对于被称为工具的代理，主管会根据工具模式确定输入。此外，LangGraph 允许在运行时将状态传递给各个工具，因此下级代理可以根据需要访问父级状态。

#### 在消息中显示代理名称¶
指明特定 AI 消息来自哪个代理会很有帮助，尤其是在消息历史记录较长的情况下。一些 LLM 提供商（例如 OpenAI）支持name在消息中添加参数——您可以使用该参数将代理名称附加到消息中。如果不支持该功能，您可以考虑手动将代理名称注入消息内容中，例如<agent>alice</agent><message>message from alice</message>。

### 在消息历史记录中表示交接¶
切换通常通过 LLM 调用专用切换工具来完成。这表示为带有工具调用的AI 消息，该消息会传递给下一个代理（LLM）。大多数 LLM 提供商不支持接收带有工具调用但没有相应工具消息的 AI 消息。

因此您有两个选择：

- 在消息列表中添加额外的工具消息，例如“已成功转移给代理 X”
- 使用工具调用删除 AI 消息


在实践中，我们发现大多数开发人员选择选项 (1)。

### 子代理的状态管理¶
一种常见的做法是让多个代理在共享消息列表上进行通信，但只将其最终的消息添加到列表中。这意味着任何中间消息（例如，工具调用）都不会保存在此列表中。

如果您确实想保存这些消息，以便将来调用这个特定的子代理时可以将它们传回，该怎么办？

有两种高级方法可以实现这一目标：

- 将这些消息存储在共享消息列表中，但在将其传递给子代理 LLM 之前对其进行过滤。例如，您可以选择过滤掉来自其他代理的所有工具调用。
- 在子代理的图表状态中为每个代理（例如alice_messages）存储单独的消息列表。这将是他们对消息历史记录的“视图”。


## 使用不同的状态模式¶
一个代理可能需要与其他代理拥有不同的状态模式。例如，搜索代理可能只需要跟踪查询和检索到的文档。在 LangGraph 中，有两种方法可以实现这一点：

- 使用单独的状态模式定义子图代理。如果子图和父图之间没有共享的状态键（通道），则需要添加输入/输出转换，以便父图知道如何与子图通信。
- 使用与整体图状态模式不同的私有输入状态模式来定义代理节点函数。这允许传递仅执行特定代理所需的信息。

# 多代理

如果单个代理需要专注于多个领域或管理许多工具，可能会遇到困难。为了解决这个问题，您可以将代理拆分成多个更小、独立的代理，并将它们组合成一个多代理系统。

在多智能体系统中，智能体之间需要进行通信。它们通过切换（handoff）来实现这一点。切换是一个原语，用于描述将控制权移交给哪个智能体以及要发送给该智能体的有效载荷。

两种最流行的多代理架构是：

- 主管——各个代理由中央主管代理协调。主管控制所有通信流程和任务委托，并根据当前上下文和任务需求决定调用哪个代理。
- swarm——智能体会根据各自的专长动态地相互交接控制权。系统会记住哪个智能体最后处于活动状态，确保在后续交互中能够继续与该智能体进行对话。

## 导师

<img src="https://langchain-ai.github.io/langgraph/agents/assets/supervisor.png">

使用langgraph-supervisor库创建一个主管多代理系统：


`    pip install langgraph-supervisor`

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor

def book_hotel(hotel_name: str):
    """Book a hotel"""
    return f"Successfully booked a stay at {hotel_name}."

def book_flight(from_airport: str, to_airport: str):
    """Book a flight"""
    return f"Successfully booked a flight from {from_airport} to {to_airport}."

flight_assistant = create_react_agent(
    model="openai:gpt-4o",
    tools=[book_flight],
    prompt="You are a flight booking assistant",
    name="flight_assistant"
)

hotel_assistant = create_react_agent(
    model="openai:gpt-4o",
    tools=[book_hotel],
    prompt="You are a hotel booking assistant",
    name="hotel_assistant"
)

supervisor = create_supervisor(
    agents=[flight_assistant, hotel_assistant],
    model=ChatOpenAI(model="gpt-4o"),
    prompt=(
        "You manage a hotel booking assistant and a"
        "flight booking assistant. Assign work to them."
    )
).compile()

for chunk in supervisor.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "book a flight from BOS to JFK and a stay at McKittrick Hotel"
            }
        ]
    }
):
    print(chunk)
    print("\n")

## 群体

<img src="https://langchain-ai.github.io/langgraph/agents/assets/swarm.png">

使用langgraph-swarm库创建一个群体多智能体系统：

`pip install langgraph-swarm`

API 参考：create_react_agent | create_swarm | create_handoff_tool

In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph_swarm import create_swarm, create_handoff_tool

transfer_to_hotel_assistant = create_handoff_tool(
    agent_name="hotel_assistant",
    description="Transfer user to the hotel-booking assistant.",
)
transfer_to_flight_assistant = create_handoff_tool(
    agent_name="flight_assistant",
    description="Transfer user to the flight-booking assistant.",
)

flight_assistant = create_react_agent(
    model="anthropic:claude-3-5-sonnet-latest",
    tools=[book_flight, transfer_to_hotel_assistant],
    prompt="You are a flight booking assistant",
    name="flight_assistant"
)
hotel_assistant = create_react_agent(
    model="anthropic:claude-3-5-sonnet-latest",
    tools=[book_hotel, transfer_to_flight_assistant],
    prompt="You are a hotel booking assistant",
    name="hotel_assistant"
)

swarm = create_swarm(
    agents=[flight_assistant, hotel_assistant],
    default_active_agent="flight_assistant"
).compile()

for chunk in swarm.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "book a flight from BOS to JFK and a stay at McKittrick Hotel"
            }
        ]
    }
):
    print(chunk)
    print("\n")

## 交接¶
多智能体交互中一个常见的模式是切换，即一个智能体将控制权移交给另一个智能体。切换允许你指定：

目的地：导航到的目标代理
有效载荷：传递给该代理的信息
这既可用于langgraph-supervisor（主管移交给个别代理），也可用于langgraph-swarm（个别代理可以移交给其他代理）。

要使用 `create_react_agent`实现切换，您需要：

1. 创建一个可以将控制权转移到不同代理的特殊工具

In [ ]:
def transfer_to_bob():
    """Transfer to bob."""
    return Command(
        # name of the agent (node) to go to
        goto="bob",
        # data to send to the agent
        update={"messages": [...]},
        # indicate to LangGraph that we need to navigate to
        # agent node in a parent graph
        graph=Command.PARENT,
    )

2. 创建可以访问交接工具的个人代理：




In [ ]:
flight_assistant = create_react_agent(
    ..., tools=[book_flight, transfer_to_hotel_assistant]
)
hotel_assistant = create_react_agent(
    ..., tools=[book_hotel, transfer_to_flight_assistant]
)

3. 定义包含各个代理作为节点的父图：

In [ ]:
from langgraph.graph import StateGraph, MessagesState
multi_agent_graph = (
    StateGraph(MessagesState)
    .add_node(flight_assistant)
    .add_node(hotel_assistant)
    ...
)

综上所述，您可以实现一个包含两个代理（航班预订助理和酒店预订助理）的简单多代理系统：

API 参考：工具| InjectedToolCallId | create_react_agent | InjectedState | StateGraph |开始|命令

In [ ]:
from typing import Annotated
from langchain_core.tools import tool, InjectedToolCallId
from langgraph.prebuilt import create_react_agent, InjectedState
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.types import Command

def create_handoff_tool(*, agent_name: str, description: str | None = None):
    name = f"transfer_to_{agent_name}"
    description = description or f"Transfer to {agent_name}"

    @tool(name, description=description)
    def handoff_tool(
        state: Annotated[MessagesState, InjectedState], 
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> Command:
        tool_message = {
            "role": "tool",
            "content": f"Successfully transferred to {agent_name}",
            "name": name,
            "tool_call_id": tool_call_id,
        }
        return Command(  
            goto=agent_name,  
            update={"messages": state["messages"] + [tool_message]},  
            graph=Command.PARENT,  
        )
    return handoff_tool

# Handoffs
transfer_to_hotel_assistant = create_handoff_tool(
    agent_name="hotel_assistant",
    description="Transfer user to the hotel-booking assistant.",
)
transfer_to_flight_assistant = create_handoff_tool(
    agent_name="flight_assistant",
    description="Transfer user to the flight-booking assistant.",
)

# Simple agent tools
def book_hotel(hotel_name: str):
    """Book a hotel"""
    return f"Successfully booked a stay at {hotel_name}."

def book_flight(from_airport: str, to_airport: str):
    """Book a flight"""
    return f"Successfully booked a flight from {from_airport} to {to_airport}."

# Define agents
flight_assistant = create_react_agent(
    model="anthropic:claude-3-5-sonnet-latest",
    tools=[book_flight, transfer_to_hotel_assistant],
    prompt="You are a flight booking assistant",
    name="flight_assistant"
)
hotel_assistant = create_react_agent(
    model="anthropic:claude-3-5-sonnet-latest",
    tools=[book_hotel, transfer_to_flight_assistant],
    prompt="You are a hotel booking assistant",
    name="hotel_assistant"
)

# Define multi-agent graph
multi_agent_graph = (
    StateGraph(MessagesState)
    .add_node(flight_assistant)
    .add_node(hotel_assistant)
    .add_edge(START, "flight_assistant")
    .compile()
)

# Run the multi-agent graph
for chunk in multi_agent_graph.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "book a flight from BOS to JFK and a stay at McKittrick Hotel"
            }
        ]
    }
):
    print(chunk)
    print("\n")

# 构建多智能体系统

如果单个代理需要专注于多个领域或管理多个工具，可能会遇到困难。为了解决这个问题，您可以将代理拆分成多个更小、独立的代理，并将它们组合成一个多代理系统。

在多智能体系统中，智能体之间需要进行通信。它们通过切换（handoff）来实现这一点。切换是一个原语，用于描述将控制权移交给哪个智能体以及要发送给该智能体的有效载荷。

本指南涵盖以下内容：

- 实现代理之间的交接
- 使用切换和预建代理来构建自定义多代理系统

要开始构建多智能体系统，请查看 LangGraph对两种最流行的多智能体架构（supervisor和swarm）的预构建实现。

## 交接¶
要在多代理系统中建立代理之间的通信，可以使用（handoff）切换 ——一种一个代理将控制权移交给另一个代理的模式。切换允许您指定：

- 目的地：要导航到的目标代理（例如，要前往的 LangGraph 节点的名称）
- 有效载荷：传递给该代理的信息（例如，状态更新）

## 创建交接¶
为了实现切换，您可以Command从代理节点或工具返回对象：

API 参考：工具| InjectedToolCallId | create_react_agent | InjectedState | StateGraph |开始|命令

In [ ]:
from typing import Annotated
from langchain_core.tools import tool, InjectedToolCallId
from langgraph.prebuilt import create_react_agent, InjectedState
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.types import Command

def create_handoff_tool(*, agent_name: str, description: str | None = None):
    name = f"transfer_to_{agent_name}"
    description = description or f"Transfer to {agent_name}"

    @tool(name, description=description)
    def handoff_tool(
        state: Annotated[MessagesState, InjectedState], 
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> Command:
        tool_message = {
            "role": "tool",
            "content": f"Successfully transferred to {agent_name}",
            "name": name,
            "tool_call_id": tool_call_id,
        }
        return Command(  
            goto=agent_name,  
            update={"messages": state["messages"] + [tool_message]},  
            graph=Command.PARENT,  
        )
    return handoff_tool

## 控制代理输入¶
您可以使用Send()原语在切换期间直接向工作代理发送数据。例如，您可以请求调用代理填充下一个代理的任务描述：



In [ ]:
from typing import Annotated
from langchain_core.tools import tool, InjectedToolCallId
from langgraph.prebuilt import InjectedState
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.types import Command, Send

def create_task_description_handoff_tool(
    *, agent_name: str, description: str | None = None
):
    name = f"transfer_to_{agent_name}"
    description = description or f"Ask {agent_name} for help."

    @tool(name, description=description)
    def handoff_tool(
        # this is populated by the calling agent
        task_description: Annotated[
            str,
            "Description of what the next agent should do, including all of the relevant context.",
        ],
        # these parameters are ignored by the LLM
        state: Annotated[MessagesState, InjectedState],
    ) -> Command:
        task_description_message = {"role": "user", "content": task_description}
        agent_input = {**state, "messages": [task_description_message]}
        return Command(
            goto=[Send(agent_name, agent_input)],
            graph=Command.PARENT,
        )

    return handoff_tool

请参阅多代理主管Send()示例，了解切换使用的完整示例。

## 构建多代理系统¶
您可以在使用 LangGraph 构建的任何代理中使用 Handoff。我们建议使用预构建代理或ToolNode，因为它们原生支持返回 Handoff 的 Handoff 工具Command。以下示例展示了如何使用 Handoff 实现一个用于预订旅行的多代理系统：



In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, START, MessagesState

def create_handoff_tool(*, agent_name: str, description: str | None = None):
    # same implementation as above
    ...
    return Command(...)

# Handoffs
transfer_to_hotel_assistant = create_handoff_tool(agent_name="hotel_assistant")
transfer_to_flight_assistant = create_handoff_tool(agent_name="flight_assistant")

# Define agents
flight_assistant = create_react_agent(
    model="anthropic:claude-3-5-sonnet-latest",
    tools=[..., transfer_to_hotel_assistant],
    name="flight_assistant"
)
hotel_assistant = create_react_agent(
    model="anthropic:claude-3-5-sonnet-latest",
    tools=[..., transfer_to_flight_assistant],
    name="hotel_assistant"
)

# Define multi-agent graph
multi_agent_graph = (
    StateGraph(MessagesState)
    .add_node(flight_assistant)
    .add_node(hotel_assistant)
    .add_edge(START, "flight_assistant")
    .compile()
)

## 多轮对话¶
用户可能希望与一个或多个代理进行多轮对话。为了构建一个能够处理此类情况的系统，您可以创建一个节点，该节点使用interrupt来收集用户输入并路由回活动代理。

然后可以将代理实现为图中的节点，执行代理步骤并确定下一步操作：
- 等待用户输入以继续对话，或者
- 通过切换路由到另一个代理（或返回自身，例如在循环中）

In [ ]:
def human(state) -> Command[Literal["agent", "another_agent"]]:
    """A node for collecting user input."""
    user_input = interrupt(value="Ready for user input.")

    # Determine the active agent.
    active_agent = ...

    ...
    return Command(
        update={
            "messages": [{
                "role": "human",
                "content": user_input,
            }]
        },
        goto=active_agent
    )

def agent(state) -> Command[Literal["agent", "another_agent", "human"]]:
    # The condition for routing/halting can be anything, e.g. LLM tool call / structured output, etc.
    goto = get_next_agent(...)  # 'agent' / 'another_agent'
    if goto:
        return Command(goto=goto, update={"my_state_key": "my_state_value"})
    else:
        return Command(goto="human") # Go to human node